# Module 17 — Stacking Blocks into nanoGPT

Module 16 built one transformer block. A GPT is: a token embedding
(Module 08) plus a positional embedding (Module 12), fed through a **stack**
of transformer blocks (Module 16), a final layer norm (Module 13), and a
linear head projecting back to vocabulary-sized logits. That's the entire
architecture — nothing else. This module assembles it, and generalizes
Module 16's pieces to handle **batches** of sequences (needed for Module
18's real training), re-verifying everything still works correctly.

## 1. Copied-in pieces from Module 16, generalized for batching

Module 16's `split_heads`/`merge_heads` assumed a single sequence
`(seq_len, d_model)`. Training needs mini-batches: `(batch, seq_len,
d_model)`. The generalization below handles *any* number of leading batch
dimensions — 0 (Module 16's original case) or 1 (batched) — since
`scaled_dot_product_attention` itself already only ever touches the last
two dimensions.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


def split_heads(t, num_heads):
    *batch_dims, seq_len, d_model = t.shape
    d_k = d_model // num_heads
    t = t.view(*batch_dims, seq_len, num_heads, d_k)
    return t.transpose(-3, -2)  # (..., num_heads, seq_len, d_k)


def merge_heads(t):
    *batch_dims, num_heads, seq_len, d_k = t.shape
    t = t.transpose(-3, -2).contiguous()
    return t.view(*batch_dims, seq_len, num_heads * d_k)


# Sanity check the generalization against Module 16's un-batched case
torch.manual_seed(42)
x_single = torch.randn(6, 16)  # (seq_len, d_model), no batch dim - Module 16's case
x_batched = x_single.unsqueeze(0)  # (1, seq_len, d_model)

split_single = split_heads(x_single, num_heads=4)
split_batched = split_heads(x_batched, num_heads=4)
assert torch.allclose(split_single, split_batched.squeeze(0))
assert torch.allclose(merge_heads(split_single), x_single)
assert torch.allclose(merge_heads(split_batched), x_batched)
print("Batched generalization matches the un-batched Module 16 behavior exactly.")

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal=True):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=causal)
        return self.Wo(merge_heads(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.ln1(x), causal=True)
        x = x + self.ffn(self.ln2(x))
        return x


print("Batched attention, feed-forward, and transformer block ready.")

## 2. The full architecture

Token embedding + positional embedding, `num_layers` stacked blocks, a
final layer norm, and a linear head back to vocabulary logits. The head's
weight is **tied** to the token embedding's weight (`self.head.weight =
self.token_embed.weight`) — the same matrix is used both to look up an
input token's vector and to score every vocabulary word against the final
hidden state. This is a real GPT-2 detail, not a simplification: it roughly
halves the parameters spent on the (often huge) vocabulary dimension, since
those parameters do double duty.

In [ ]:
class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, d_ff=None):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight  # weight tying

    def forward(self, idx):
        seq_len = idx.shape[-1]
        assert seq_len <= self.max_seq_len
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)


torch.manual_seed(42)
vocab_size, d_model, num_heads, num_layers, max_seq_len = 27, 32, 4, 3, 32
model = NanoGPT(vocab_size, d_model, num_heads, num_layers, max_seq_len)

n_params_tied = sum(p.numel() for p in model.parameters())
n_params_if_untied = n_params_tied + vocab_size * d_model
print(f"Parameters with weight tying:    {n_params_tied:,}")
print(f"Parameters if head were untied:  {n_params_if_untied:,}")
assert model.head.weight is model.token_embed.weight
print("\nConfirmed: head.weight and token_embed.weight are the literal same tensor.")

## 3. A batched forward pass

In [ ]:
batch_size, seq_len = 3, 10
idx = torch.randint(0, vocab_size, (batch_size, seq_len))
logits = model(idx)
print("input shape: ", idx.shape)
print("output shape:", logits.shape, "(batch, seq_len, vocab_size) - one distribution per position")
assert logits.shape == (batch_size, seq_len, vocab_size)

## 4. The whole model is still correctly causal

Same test as Module 16, now through embeddings + every stacked block +
final norm + head: perturbing a later token must never change an earlier
position's logits.

In [ ]:
idx_original = torch.randint(0, vocab_size, (1, seq_len))
logits_original = model(idx_original)

idx_modified = idx_original.clone()
idx_modified[0, 4] = (idx_modified[0, 4] + 1) % vocab_size  # change token at position 4

logits_modified = model(idx_modified)

for pos in range(seq_len):
    changed = not torch.allclose(logits_original[0, pos], logits_modified[0, pos], atol=1e-6)
    expected = pos >= 4
    assert changed == expected, f"position {pos}: changed={changed}, expected={expected}"

print("Confirmed: the full model - embeddings, every block, the head - is correctly causal end to end.")

## 5. Generating from an untrained model

The generation loop itself (predict next-token distribution, sample,
append, repeat) works right now, before any training — Module 18 trains
this same architecture on real text.

In [ ]:
vocab = ["."] + list("abcdefghijklmnopqrstuvwxyz")
stoi = {ch: i for i, ch in enumerate(vocab)}

@torch.no_grad()
def generate(model, start_idx, max_new_tokens):
    idx = start_idx
    for _ in range(max_new_tokens):
        context = idx[:, -model.max_seq_len:]
        logits = model(context)
        next_logits = logits[:, -1, :]
        probs = F.softmax(next_logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return idx


start = torch.tensor([[stoi["."]]])
generated = generate(model, start, max_new_tokens=20)
print("".join(vocab[i] for i in generated[0].tolist()))

An untrained model isn't just noisy — this particular architecture has a
specific, explainable bias at initialization. Check the top predicted next
token starting from a few different single characters:

In [ ]:
for ch in [".", "k", "x", "q"]:
    start_tok = torch.tensor([[stoi[ch]]])
    with torch.no_grad():
        probs = F.softmax(model(start_tok)[:, -1, :], dim=-1)
    top = probs[0].argmax().item()
    print(f"start={ch!r:>4}  top predicted next={vocab[top]!r:>4}  prob={probs[0, top].item():.4f}")

print("\nEven with zero training, the model strongly \'predicts\' whatever token it was just given, with near-certainty. This isn\'t noise - it\'s a structural side effect of this exact architecture: residual connections (Module 14) carry the input embedding straight through every block mostly unchanged, and weight tying (step 2 above) means the output head scores every candidate by comparing against that SAME embedding table - so a token's own embedding is, at initialization, its own best match. Training is what teaches the model to override this default with something that actually depends on context.")

## Recap

- `NanoGPT` = token embedding + positional embedding → N transformer
  blocks → final layer norm → weight-tied linear head. That's the entire
  architecture behind GPT-2 and every model this project builds toward,
  just at a much smaller scale.
- Generalized Module 16's attention helpers to handle batches, and
  re-verified every correctness property (causality, shape) still holds.
- The forward pass and generation loop both work mechanically — what's
  missing is training, which is Module 18.